# Юдин Артём М4121

In [2]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [3]:
X_train = pd.read_csv("../data/preprocessed/X_train_smart_regex_7020samples.csv")
X_val = pd.read_csv("../data/preprocessed/X_val_smart_regex_1980samples.csv")

y_train = pd.read_csv("../data/preprocessed/y_train_7020samples.csv")
y_val = pd.read_csv("../data/preprocessed/y_val_1980samples.csv")

# объединение токенов вместе
for df in (X_train, X_val):
    df["untokenized_ttext"] = df["ttext"].apply(lambda x: " ".join(eval(x)))

df = 5e-4
# инициализация векторизаторов
bow_vectorizer = CountVectorizer(min_df=df, max_df=1 - df)
tdidf_vectorizer = TfidfVectorizer(min_df=df, max_df=1 - df)

# инициализируем скейлеры
bow_scaler = StandardScaler()
tfidf_scaler = StandardScaler()

bow_pca = PCA(1360, random_state=42)
tfidf_pca = PCA(1786, random_state=42)

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

# кодирование и масштабируем данные
X_train_bow = bow_scaler.fit_transform(
    bow_vectorizer.fit_transform(X_train["untokenized_ttext"]).toarray()
)
X_val_bow = bow_scaler.transform(
    bow_vectorizer.transform(X_val["untokenized_ttext"]).toarray()
)

X_train_tfidf = tfidf_scaler.fit_transform(
    tdidf_vectorizer.fit_transform(X_train["untokenized_ttext"]).toarray()
)
X_val_tfidf = tfidf_scaler.transform(
    tdidf_vectorizer.transform(X_val["untokenized_ttext"]).toarray()
)

/home/artyom/myprojects/ITMO/IIML/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/artyom/myprojects/ITMO/IIML/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


In [4]:
def train(
    X_train_bow, X_train_tfidf, y_train, X_val_bow, X_val_tfidf, y_val
) -> dict[str, list]:
    metrics = {
        "Accuracy": accuracy_score,
        "Precision": precision_score,
        "Recall": recall_score,
        "F1": f1_score,
    }

    models = {
        "Logistic Regression": LogisticRegression(random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42, min_samples_leaf=3),
        "KNN": KNeighborsClassifier(weights="distance"),
        "BernoulliNB": BernoulliNB(),
        "GaussianNB": GaussianNB(),
        "SVC": SVC(),
    }
    df_dict = {
        "Model": [],
        "Encoding": [],
        "Accuracy": [],
        "Precision": [],
        "Recall": [],
        "F1": [],
    }

    for model_name, model in models.items():
        df_dict["Model"].append(model_name)
        df_dict["Encoding"].append("BoW")

        # обучаем модель
        model.fit(X_train_bow, y_train)
        y_pred = model.predict(X_val_bow)

        # рассчитываем метрики для обученного набора
        for metric_name, metric in metrics.items():
            df_dict[metric_name].append(round(metric(y_val, y_pred), 2))

        df_dict["Model"].append(model_name)
        df_dict["Encoding"].append("TF-IDF")

        # обучаем модель
        model.fit(X_train_tfidf, y_train)
        y_pred = model.predict(X_val_tfidf)

        # рассчитываем метрики для обученного набора
        for metric_name, metric in metrics.items():
            df_dict[metric_name].append(round(metric(y_val, y_pred), 2))

    return df_dict

In [5]:
df = pd.DataFrame(
    train(X_train_bow, X_train_tfidf, y_train, X_val_bow, X_val_tfidf, y_val)
)
df

,Model,Encoding,Accuracy,Precision,Recall,F1
0,Logistic Regression,BoW,0.63,0.63,0.61,0.62
1,Logistic Regression,TF-IDF,0.63,0.64,0.61,0.62
2,Decision Tree,BoW,0.61,0.63,0.54,0.58
3,Decision Tree,TF-IDF,0.62,0.63,0.57,0.60
4,Random Forest,BoW,0.66,0.65,0.67,0.66
5,Random Forest,TF-IDF,0.67,0.66,0.69,0.68
6,KNN,BoW,0.60,0.61,0.57,0.59
7,KNN,TF-IDF,0.59,0.62,0.48,0.54
8,BernoulliNB,BoW,0.67,0.68,0.64,0.66
9,BernoulliNB,TF-IDF,0.67,0.68,0.64,0.66


# Понижение размерности

In [6]:
X_train_bow_pca = bow_pca.fit_transform(X_train_bow)
X_val_bow_pca = bow_pca.transform(X_val_bow)

X_train_tfidf_pca = tfidf_pca.fit_transform(X_train_tfidf)
X_val_tfidf_pca = tfidf_pca.transform(X_val_tfidf)

# обучение модели
df_pca = pd.DataFrame(
    train(
        X_train_bow_pca,
        X_train_tfidf_pca,
        y_train,
        X_val_bow_pca,
        X_val_tfidf_pca,
        y_val,
    )
)
df_pca

,Model,Encoding,Accuracy,Precision,Recall,F1
0,Logistic Regression,BoW,0.65,0.66,0.63,0.64
1,Logistic Regression,TF-IDF,0.64,0.64,0.62,0.63
2,Decision Tree,BoW,0.55,0.55,0.54,0.54
3,Decision Tree,TF-IDF,0.54,0.53,0.55,0.54
4,Random Forest,BoW,0.61,0.62,0.57,0.60
5,Random Forest,TF-IDF,0.62,0.63,0.58,0.61
6,KNN,BoW,0.60,0.61,0.56,0.58
7,KNN,TF-IDF,0.58,0.59,0.53,0.56
8,BernoulliNB,BoW,0.64,0.65,0.62,0.63
9,BernoulliNB,TF-IDF,0.66,0.66,0.64,0.65


# Выводы
- Классификация на наших данных работает плохо
- Гауссовский Наивный Байес отрабатывает на данных немногим лучше случайного угадывания
- Извлечение признаков дало прирост метрик для логистической регрессии. В остальных случаях же наблюдается ухудшение показателей
- Случайный лес на TF-IDF отрабатывает лучше всего. При этом лучше отдельных деревьев, что может говорить об эффектиности ансамблирования